In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import camelot
import pdfplumber
from googletrans import Translator

from bidi.algorithm import get_display
from arabic_reshaper import reshape

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'LB BLI' ## change to current controller name
print("LB BLI Web Scraping Tool v.1.0")

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'LB BLI SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

#Assigning the folders that are going to be used in the process
#scriptfolder = os.path.dirname(os.path.abspath(__file__))
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)


LB BLI Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------
mainUrl = 'https://www.bdl.gov.lb/institutions.php'


Typology ={

            'LB BLI 1': 'Banks',
            'LB BLI 2': 'Financial Institutions',
            'LB BLI 3': "Foreign Banks' Representation Offices in Lebanon",
            'LB BLI 4': 'Detailed list of Exchange Institutions',

            
}

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')


In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict




# Initialize the translator
translator = Translator()
def translate_text(text):
    return translator.translate(text, src='ar', dest='en').text



In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

driver.get(mainUrl)
sleep(3)

soup = BeautifulSoup(driver.page_source, "html.parser")
bloc = soup.find("div",{"class":"allcontent"})
form1 = bloc.find("form",{"id":"form1"})

trs = bloc.find_all('tr')
key = ''
for tr in trs:

    inistitution_name = tr.find('td')
    if inistitution_name is not None:
        
        key = next((k for k, v in Typology.items() if v == inistitution_name.text), '')
        
    if key!='':
        print(key)
        print(inistitution_name.text)
        linkes = tr.find_all('a',class_="engbut")
        document_url = ''
        if linkes:
            document_url = 'https://www.bdl.gov.lb/'+ linkes[0]['href']
        else:
            linkes = tr.find_all('a',class_="arbut")
            if linkes:
                document_url = 'https://www.bdl.gov.lb/'+ linkes[0]['href']
        
        if document_url:
            sleep(3)
            driver.get(document_url)
            sleep(2)

        pdf_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder, pdf_file)
        sleep(2)

        if key == 'LB BLI 1':

            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            for table in tables:
                for tab in table:
                    
                    try:
                        if tab[0].isdigit():
                            id_ = tab[0]

                            name = tab[-1]
                            #print(id_,name)
                            sqldict['Name'].append(name)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['Cntry'].append('LB')
                            sqldict['RegulationType'].append('Regulated')
                            sqldict['RegCtry'].append('LB')
                            sqldict['RegCode'].append('BLI')
                            sqldict['ListCode'].append(key.split()[-1])
                            sqldict = bourange_same_length_array(sqldict)
                    except:
                        pass

        elif  key == 'LB BLI 2' or key == 'LB BLI 3':
            sleep(3)

            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            for table in tables:
                for tab in table:
                    
                    if tab[0] == 'NO.':
                        continue
                    name = tab[-1]
                    sqldict['Name'].append(name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Cntry'].append('LB')
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append('LB')
                    sqldict['RegCode'].append('BLI')
                    sqldict['ListCode'].append(key.split()[-1])
                    sqldict = bourange_same_length_array(sqldict)

        elif key == 'LB BLI 4':
            tables = camelot.read_pdf(filePath, pages='all',shift_text=[''])     
            for k in range(tables.n):
                df_Table = tables[k].df
                rows, cols = df_Table.shape
                
                try:
                    id_ = df_Table.iloc[:, -1]    # Last column
                    name_ = df_Table.iloc[:, -2]  # Second-to-last column
                    tel_ = df_Table.iloc[:, -0] 
                    if cols == 12:
                        address_ = df_Table.iloc[:, -7]
                    else:
                        address_ = df_Table.iloc[:, -4]
                    # print(list(df_Table.columns))

                    for idx, val in id_.items():
                        # Check if ID can be converted to digit
                        if str(val).isdigit():
                            #print(f"Row {idx}: ID={val}, Name={name_[idx].replace('\n',' ')}, Address_ = {address_[idx].replace('\n',' ')}, Tel = {tel_[idx].replace('\n',' ')}")
                            ori_name = name_[idx].replace('\n',' ')
                            
                            sleep(3)
                            reshaped_text = reshape(ori_name)
                            bidi_name = get_display(reshaped_text)
                            #print(bidi_text)
                            sleep(3)
                            trans_name = translate_text(bidi_name)
                            # print(trans_name)
                            sqldict['Name'].append(bidi_name)

                            ori_address = address_[idx].replace('\n',' ')
                            # reshaped_text_address = reshape(ori_address)
                            # bidi_address = get_display(reshaped_text_address)
                            # trans_lang = translate_text(bidi_address)
                            # print(ori_address, ' Translated to: ', trans_lang)
                            sqldict['Address_1'].append(ori_address)
                            sqldict['Name - Mother Company'].append(trans_name)
                            #sqldict['Address_1'].append(address_[idx].replace('\n',' '))
                            sqldict['Phone'].append(tel_[idx].replace('\n',' '))
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['Cntry'].append('LB')
                            sqldict['RegulationType'].append('Regulated')
                            sqldict['RegCtry'].append('LB')
                            sqldict['RegCode'].append('BLI')
                            sqldict['ListCode'].append(key.split()[-1])
                            sqldict['InternalID_1_type'].append('Number')
                            sqldict['InternalID_1'].append(val)
                            sqldict = bourange_same_length_array(sqldict)
                except:
                    pass
            
    if os.path.exists(tempfolder):
        for temp_file in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, temp_file))

LB BLI 1
Banks
LB BLI 2
Financial Institutions
LB BLI 3
Foreign Banks' Representation Offices in Lebanon
LB BLI 4
Detailed list of Exchange Institutions


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)
writer.save()
writer.close()
driver.quit()
sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_27924\802773129.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df.to_excel('list_4_reverse_font.xlsx')